In [1]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import scipy
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
import numpy as np

from statsmodels.discrete.conditional_models import ConditionalLogit

from preproces_prod3 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data, comparar_medias_test,analyze_vrs_data, integer_programming_matching_gurobi, match_nn_max_dist_weigths, charly_mip, charly_double_mip, mylogit

from IPython.core.display import display
warnings.filterwarnings("ignore")

In [2]:
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [3]:
icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
]

In [4]:
df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_s39_2012.csv', index_col=0)
df_upc_match_case = pd.read_csv(path_data/'df_upc_match_case_s39_2012.csv', index_col=0)

In [ ]:
df = df_vrs_match_case.copy().query('is_rural==1')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()

print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

weights = {'edad_relativa': 1,'PESO': 0.4} 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','PREVI','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=5,
                                                              weights=weights)

results = mylogit(matched_data)

distance_vars = ['PESO','edad_relativa','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")

results = mylogit(matched_data)

N° de obs case 43 y control 7916
Total cases = 43, Total controls = 7916
Total cases matched is : 33, Total control matched is : 99
ratio: 1:3
No matched : 10

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion     -2.531596  0.079532    92.046798  98.291961  62.967234

IC disntace: 3.076457057307208
creacion conjuntos A_i time: 3.310112714767456
creacion variables time: 28.418200969696045
optimize model 1 time: 0.04098176956176758
optimize model 2 time: 0.17702174186706543
matched_data time: 0.2028489112854004
Total cases = 43, Total controls = 7916
Total cases matched is : 43, Total control matched is : 129
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion     -1.891695  0.150816    84.918403  95.323896  51.358104

IC disntace: 2.3420199552668715


In [12]:
df = df_vrs_match_case.copy().query('group=="CATCH_UP"')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()

print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

weights = {'edad_relativa': 1,'PESO': 0.4} 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','PREVI','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=5,
                                                              weights=weights)

results = mylogit(matched_data)

distance_vars = ['PESO','edad_relativa','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group','PREVI'], ratio="1:3")

results = mylogit(matched_data)

N° de obs case 705 y control 78478
Total cases = 705, Total controls = 78478
Total cases matched is : 416, Total control matched is : 1248
ratio: 1:3
No matched : 289

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion     -2.876137  0.056352    94.364799  96.192158  91.660502

IC disntace: 0.7839404885329051
creacion conjuntos A_i time: 277.6945722103119
creacion variables time: 1793.2152004241943
optimize model 1 time: 6.51989221572876
optimize model 2 time: 13.51351022720337
matched_data time: 5.50214409828186
Total cases = 705, Total controls = 78478
Total cases matched is : 641, Total control matched is : 1923
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion     -2.448458  0.086427    91.357324  93.451854  88.592825

IC disntace: 0.5550606263984195


In [ ]:
df = df_upc_match_case.copy().query('is_rural==1')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()

print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

weights = {'edad_relativa': 1,'PESO': 0.4} 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=5,
                                                              weights=weights)

results = mylogit(matched_data)

distance_vars = ['PESO','edad_relativa','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")

results = mylogit(matched_data)

N° de obs case 11 y control 7948
Total cases = 11, Total controls = 7948
Total cases matched is : 10, Total control matched is : 30
ratio: 1:3
No matched : 1

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975     0.025
estado_inmunizacion     -1.791395  0.166727     83.32726  98.487952 -83.84356

IC disntace: estado_inmunizacion    4.80062
dtype: float64
creacion conjuntos A_i time: 0.44146251678466797
creacion variables time: 2.8112001419067383
optimize model 1 time: 0.002998828887939453
optimize model 2 time: 0.020000457763671875
matched_data time: 0.01902461051940918
Total cases = 11, Total controls = 7948
Total cases matched is : 11, Total control matched is : 33
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion      -1.79128  0.166747    83.325344  98.487709 -83.856237

IC disntace: estado_inmunizacion    4.800528
dtype: float64


In [30]:
df_vrs_match_case.group.unique()

array(['CATCH_UP', 'SEASONAL'], dtype=object)

In [6]:
df = df_upc_match_case.copy().query('is_poor==1').query('32<=SEMANAS<=36')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()

print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

weights = {'edad_relativa': 1,'PESO': 0.4} 

# matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
#                                                               match_vars_nn=['edad_relativa','PESO'], 
#                                                               match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','prematuro'],
#                                                               match_vars_onehot=[],
#                                                               ratio="1:15",
#                                                               max_distance=5,
#                                                               weights=weights)

# results = mylogit(matched_data)

distance_vars = ['PESO','edad_relativa','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group','PREVI'], ratio="1:3")

results = mylogit(matched_data)


N° de obs case 25 y control 6615
creacion conjuntos A_i time: 0.7642788887023926
creacion variables time: 5.092067241668701
optimize model 1 time: 0.0069882869720458984
optimize model 2 time: 0.016003847122192383
matched_data time: 0.04300713539123535
Total cases = 25, Total controls = 6615
Total cases matched is : 24, Total control matched is : 72
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion     -3.295775  0.037039    96.296066  99.530711  70.766143

IC disntace: 4.131864061550926


In [ ]:
df = df_vrs_match_case.copy().query('32<=SEMANAS<=36')
regions = df.region.unique()

weights = {'edad_relativa': 1,'PESO': 0.4} 
distance_vars = ['PESO','edad_relativa','SEMANAS']

regions_prematuros = {}

for region in regions:
    if pd.isnull(region):
        continue
    print(region)
    df_region = df.query(f'region=="{region}"')
    df_case= df_region.query("diag_vrs==True")
    df_control= df_region.query("diag_vrs==False")
    
    matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','region','muy_prematuro','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=5,
                                                              weights=weights)

    results = mylogit(matched_data)
    if results == 'No hay no inmunizados':
        regions_prematuros[region] = f"{region}: match_nn___ No hay no inmunizados"
    else:
        regions_prematuros[region] = f"{region}: match_nn___ efectividad = {results['efectividad']}, ic_dist = {results['IC_distance']}"

    matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")

    results = mylogit(matched_data)
    if results == 'No hay no inmunizados':
        regions_prematuros[region] = f"{region}: MIP___ No hay no inmunizados"
    else:
        regions_prematuros[region] = f"{region}: MIP___ efectividad = {results['efectividad']}, ic_dist = {results['IC_distance']}"

In [38]:
df = df_vrs_match_case.copy().query('32<=SEMANAS<=36').query('group=="SEASONAL"')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
display(df_case.inmunizado.value_counts())
display(df_control.inmunizado.value_counts())

inmunizado
1    81
Name: count, dtype: int64

inmunizado
1    6195
0     203
Name: count, dtype: int64

In [40]:
df = df_vrs_match_case.copy().query('32<=SEMANAS<=36').query('group=="CATCH_UP"')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
display(df_case.inmunizado.value_counts())
display(df_control.inmunizado.value_counts())

inmunizado
1    66
0    30
Name: count, dtype: int64

inmunizado
1    6309
0     441
Name: count, dtype: int64

In [25]:
df_upc_match_case.columns[:100]

Index(['RUN', 'RUN_RNI', 'RUN_MADRE', 'VACUNADO', 'MARCA', 'FECHA_NACIMIENTO',
       'MES_NAC', 'ANO_NAC', 'SEXO', 'SEMANAS', 'PESO', 'TALLA', 'EDAD_M',
       'INS_C_M', 'INS_N_M', 'COMUNA', 'COMUNA_N', 'REG_RES', 'URBA_RURAL',
       'NAC_MA', 'FECHA_INMUNIZACION', 'FECHA_DEFUNCION', 'CAUSA_DEFUNCION',
       'VIVO', 'FALLECIDO_PREVIO', 'ESTAB', 'ServicioSalud', 'Seremi',
       'P_ORIGEN', 'PREVI', 'FECHA_INGRESO', 'FECHA_EGRESO', 'AREA_FUNC_I',
       'SER_CLIN_I', 'DIA_1_TRAS', 'MES_1_TRAS', 'ANO_1_TRAS', 'AREAF_1_TRAS',
       'SERC_1_TRAS', 'DIA_2_TRAS', 'MES_2_TRAS', 'ANO_2_TRAS', 'AREAF_2_TRAS',
       'SERC_2_TRAS', 'DIA_3_TRAS', 'MES_3_TRAS', 'ANO_3_TRAS', 'AREAF_3_TRAS',
       'SERC_3_TRAS', 'DIA_4_TRAS', 'MES_4_TRAS', 'ANO_4_TRAS', 'AREAF_4_TRAS',
       'SERC_4_TRAS', 'DIA_5_TRAS', 'MES_5_TRAS', 'ANO_5_TRAS', 'AREAF_5_TRAS',
       'SERC_5_TRAS', 'DIA_6_TRAS', 'MES_6_TRAS', 'ANO_6_TRAS', 'AREAF_6_TRAS',
       'SERC_6_TRAS', 'DIA_7_TRAS', 'MES_7_TRAS', 'ANO_7_TRAS', 'AR

In [27]:
df_upc_match_case.columns[100:200]

Index(['fecha_tras_1', 'fecha_tras_2', 'fecha_tras_3', 'fecha_tras_4',
       'fecha_tras_5', 'fecha_tras_6', 'fecha_tras_7', 'fecha_tras_8',
       'fecha_tras_9', 'fecha_nac', 'fechaIng_any', 'fechaEgr', 'fechaInm',
       'VRS_D1', 'VRS_D1y3', 'VRS_Dall', 'diag_irag', 'diag_ira_alta',
       'LRTI_Flag', 'LRTI_all_j', 'LRTI_Flag_Dall', 'fechaIng_vrs',
       'fechaIng_LRTI', 'fechaIng_vrs_Dall', 'fechaIng_LRTI_Dall', 'group',
       'sexo', 'prematuro_extremo', 'muy_prematuro', 'prematuro_moderado',
       'prematuro', 'atypic_mom_age', 'region', 'Macrozona2', 'cama',
       'fecha_upc', 'fecha_upc_vrs', 'fecha_upc_vrs_Dall', 'days_upc',
       'dias_en_ing', 'days_estad_vrs', 'days_estad_vrs_Dall', 'is_rural',
       'categori_macro', 'categori_regions', 'exp_rural', 'is_poor',
       'vrs_pre_campaña', 'lrti_pre_campaña', 'any_pre_campaña',
       'upc_pre_campaña', 'dias_en_area_1', 'dias_en_area_2', 'dias_en_area_3',
       'dias_en_area_4', 'dias_en_area_5', 'dias_en_area_6', '

In [104]:
df_upc_match_case.columns[200:]

Index(['inm_70d', 'inm_mayor_70d', 'inm_77d', 'inm_mayor_77d', 'inm_84d',
       'inm_mayor_84d', 'inm_91d', 'inm_mayor_91d', 'inm_98d', 'inm_mayor_98d',
       'inm_105d', 'inm_mayor_105d', 'inm_112d', 'inm_mayor_112d', 'inm_119d',
       'inm_mayor_119d', 'inm_126d', 'inm_mayor_126d', 'inm_133d',
       'inm_mayor_133d', 'inm_140d', 'inm_mayor_140d', 'inm_147d',
       'inm_mayor_147d', 'inm_154d', 'inm_mayor_154d', 'inm_161d',
       'inm_mayor_161d', 'inm_168d', 'inm_mayor_168d', 'inm_175d',
       'inm_mayor_175d', 'inm_182d', 'inm_mayor_182d', 'inmunizado',
       'chile_chico', 'estado_inmunizacion', 'diag_vrs', 'diag_1_leter',
       'edad_relativa', 'ingreso_relativo'],
      dtype='object')